# C12-classical-models — Practice p10 — Solution


Every feature contributes midpoints of adjacent distinct values. Since the parent term is common, minimizing weighted child impurity maximizes gain; the tuple key makes ties deterministic.


In [ ]:
import numpy as np

X_p10 = np.array([[0.0,0.0],[1.0,1.0],[2.0,0.0],[3.0,1.0],
                  [4.0,0.0],[5.0,1.0]], dtype=np.float64)
y_p10 = np.array([0,0,0,1,1,1], dtype=np.int64)


def best_gini_split(X, y):
    if not isinstance(X, np.ndarray) or X.dtype != np.float64 or X.ndim != 2:
        raise ValueError("X must be a float64 matrix")
    if not isinstance(y, np.ndarray) or not np.issubdtype(y.dtype, np.integer) or y.ndim != 1:
        raise ValueError("y must be an integer vector")
    if X.shape[0] < 2 or X.shape[1] < 1 or y.shape != (X.shape[0],) or np.unique(y).size < 2:
        raise ValueError("invalid shape/classes")
    if not np.isfinite(X).all():
        raise ValueError("X must be finite")
    def impurity(labels):
        counts = np.unique(labels, return_counts=True)[1].astype(np.float64)
        probabilities = counts / labels.size
        return float(1.0 - probabilities @ probabilities)
    parent = impurity(y)
    candidates = []
    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])
        for threshold in (values[:-1] + values[1:]) / 2.0:
            left = X[:, feature] <= threshold
            n_left = int(left.sum())
            if n_left == 0 or n_left == X.shape[0]:
                continue
            weighted = (n_left * impurity(y[left]) + (X.shape[0] - n_left) * impurity(y[~left])) / X.shape[0]
            gain = parent - weighted
            if gain > 0.0:
                candidates.append((float(weighted), int(feature), float(threshold), float(gain)))
    if not candidates:
        return None
    # PLAN018_MUTATION_TARGET: C12-p10-best-split
    weighted, feature, threshold, gain = min(candidates, key=lambda item: item[:3])
    return {"feature": feature, "threshold": threshold,
            "weighted_impurity": weighted, "gain": gain}


split_p10 = best_gini_split(X_p10, y_p10)


### Answer check


In [ ]:
ATOL = 1e-12
RTOL = 1e-10
# PLAN018_ANSWER_CHECK: C12-p10-best-split
assert split_p10["feature"] == 0
assert np.isclose(split_p10["threshold"], 2.5, atol=ATOL, rtol=RTOL)
assert np.isclose(split_p10["weighted_impurity"], 0.0, atol=ATOL, rtol=RTOL)
assert np.isclose(split_p10["gain"], 0.5, atol=ATOL, rtol=RTOL)
tie_X_p10 = np.array([[0.,0.],[0.,0.],[1.,1.],[1.,1.]], dtype=np.float64)
tie_y_p10 = np.array([0,0,1,1], dtype=np.int64)
tie_p10 = best_gini_split(tie_X_p10, tie_y_p10)
assert tie_p10["feature"] == 0 and np.isclose(tie_p10["threshold"], 0.5, atol=ATOL, rtol=RTOL)
